# 01 · Data pipeline

Builds an industry × year panel (≈ 65 private industries, 1997–latest) from four public APIs.

| Source | Dataset | Variables |
|---|---|---|
| BEA | GDP by Industry | Nominal and real value added; compensation of employees; gross operating surplus |
| BEA | NIPA Table 6.5D | Full-time-equivalent (FTE) employees by industry |
| BEA | Fixed Assets | Net stock, quantity index and investment of intellectual property products (software, R&D, entertainment originals); equipment; total fixed assets |
| FRED | Macro-financial | Fed funds, 10-year yield, Baa spread, VIX, NASDAQ, PCE prices, e-commerce share, recessions, BLS labour share |
| BLS | CES and productivity | Finance sub-sector employment and earnings (cross-checks) |
| Census | Business Formation Statistics | Business applications by sector (dynamism control) |

**Design choices for reproducibility**

* BEA tables are located by their *descriptions*, not hard-coded IDs, and each choice is printed and saved in `data/processed/pipeline_log.txt`.
* Every raw response is cached in `data/raw/`. Re-running the notebook uses the cache; set `REFRESH = True` to download fresh vintages.
* Industries are BEA's most detailed private industries; owner-occupied housing and government are excluded.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src" / "exgap").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
import pandas as pd
pd.set_option("display.width", 160, "display.max_columns", 40)
from exgap.build_panel import build_panel, load_panel
from exgap import config

REFRESH = False      # True = re-download everything (new data vintage)
panel = build_panel(refresh=REFRESH, end_year=config.END_YEAR)

Pipeline run 2026-11-22
BEA GDP by industry ...
  GDPbyIndustry va_nom: TableID 1 - Value Added by Industry (A) (Q)
  GDPbyIndustry va_real: TableID 10 - Real Value Added by Industry (A) (Q)
  GDPbyIndustry components: TableID 6 - Components of Value Added by Industry (A)
  GDPbyIndustry components_pct: TableID 7 - Components of Value Added by Industry as a Percentage of Value Added (A)
  selected 65 leaf private industries (4 in finance & insurance)
  labour share computed from components-of-value-added levels
BEA NIPA employment ...
  NIPA FTE: T60500D - Table 6.5D. Full-Time Equivalent Employees by Industry (A)
  [fte] matched 65/65 industries
BEA Fixed Assets ...
  FixedAssets ipp_stock: FAAt301I - Table 3.1I. Current-Cost Net Stock of Intellectual Property Products by Industry (A)
  FixedAssets ipp_qty: FAAt302I - Table 3.2I. Chain-Type Quantity Indexes for Net Stock of Intellectual Property Products by Industry (A)
  FixedAssets ipp_inv: FAAt307I - Table 3.7I. Investment in Priva

## 1. Industries in the panel

In [3]:
ind = (panel.groupby(["code", "name"])[["finance", "information", "it_producing"]].max()
          .reset_index().sort_values(["finance", "code"], ascending=[False, True]))
print(f"{len(ind)} industries, of which {ind['finance'].sum()} in finance & insurance")
ind

65 industries, of which 4 in finance & insurance


,code,name,finance,information,it_producing
43,521CI,"Federal Reserve banks, credit intermediation, ...",1,0,0
44,523,"Securities, commodity contracts, and investments",1,0,0
45,524,Insurance carriers and related activities,1,0,0
46,525,"Funds, trusts, and other financial vehicles",1,0,0
0,111CA,Farms,0,0,0
...,...,...,...,...,...
60,713,"Amusements, gambling, and recreation industries",0,0,0
61,721,Accommodation,0,0,0
62,722,Food services and drinking places,0,0,0
63,81,"Other services, except government",0,0,0


## 2. Coverage by variable

In [4]:
key_vars = ["va_nom", "va_real", "labor_share", "fte", "ipp_stock", "ipp_qty", "ipp_inv", "eq_stock",
            "fa_stock", "productivity", "pay_real", "fedfunds", "baa_spread", "vix", "ecom_share", "business_apps"]
key_vars = [v for v in key_vars if v in panel]
cov = pd.DataFrame({
    "non_missing_share": panel[key_vars].notna().mean().round(3),
    "first_year": [panel.loc[panel[v].notna(), "year"].min() for v in key_vars],
    "last_year": [panel.loc[panel[v].notna(), "year"].max() for v in key_vars],
})
cov.to_csv(config.TABLES_DIR / "data_coverage.csv")
cov

,non_missing_share,first_year,last_year
va_nom,1.000,1997,2025
va_real,1.000,1997,2025
labor_share,1.000,1997,2025
fte,0.966,1998,2025
ipp_stock,1.000,1997,2025
ipp_qty,1.000,1997,2025
ipp_inv,1.000,1997,2025
eq_stock,1.000,1997,2025
fa_stock,0.985,1997,2025
productivity,0.966,1998,2025


## 3. Sanity checks
These assertions stop the workflow if the data look wrong (for example if BEA changes a table layout).

In [5]:
problems = []
ls = panel["labor_share"].dropna()
if not ls.between(0, 1.5).mean() > 0.95:
    problems.append("labour share outside [0, 1.5] for >5% of rows")
if panel["finance"].groupby(panel["code"]).max().sum() < 3:
    problems.append("fewer than 3 finance industries")
for v, thr in [("fte", 0.8), ("ipp_stock", 0.8), ("va_real", 0.9)]:
    share = panel.groupby("code")[v].apply(lambda s: s.notna().any()).mean()
    if share < thr:
        problems.append(f"{v} matched for only {share:.0%} of industries")
print("All checks passed." if not problems else "CHECK FAILED:\n - " + "\n - ".join(problems))

print("\nLabour share by group, latest common year:")
yr = int(panel.dropna(subset=["labor_share", "ipp_intensity"])["year"].max())
print(panel[panel.year == yr].groupby("finance")[["labor_share", "ipp_intensity"]].median().round(3))

All checks passed.

Labour share by group, latest common year:
         labor_share  ipp_intensity
finance                            
0              0.559          0.111
1              0.485          0.179


## 4. Data dictionary and saved files

In [6]:
dd = pd.read_csv(config.PROCESSED_DIR / "data_dictionary.csv")
for f in sorted(config.PROCESSED_DIR.glob("*.csv")):
    print(f"  {f.name:<38} {f.stat().st_size/1024:8.1f} KB")
dd

  bea_fixedassets_tables.csv                 11.9 KB
  bea_gdpbyindustry_tables.csv                3.0 KB
  bea_industry_codes_all.csv                  3.6 KB
  bea_nipa_tables.csv                        25.1 KB
  bls_finance_checks.csv                     10.2 KB
  data_dictionary.csv                         1.9 KB
  macro_annual.csv                            2.8 KB
  panel.csv                                1253.6 KB


,variable,definition
0,code,BEA GDP-by-industry code
1,name,Industry name
2,year,Calendar year
3,va_nom,"Value added, current $ billions (BEA GDPbyIndu..."
4,va_real,"Real value added, chained (2017) $ billions (B..."
5,comp,"Compensation of employees, $ billions"
6,gos,"Gross operating surplus, $ billions"
7,labor_share,Compensation of employees / value added
8,gos_share,Gross operating surplus / value added
9,fte,"Full-time-equivalent employees, persons (BEA N..."
